In [3]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model="mixtral-8x7b-32768",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key="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"
    # other params...
)

messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content='I love programming in French is: "J\'aime programmer."', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 30, 'total_tokens': 46, 'completion_time': 0.024012196, 'prompt_time': 0.002670433, 'queue_time': 0.027870255, 'total_time': 0.026682629}, 'model_name': 'mixtral-8x7b-32768', 'system_fingerprint': 'fp_c5f20b5bb1', 'finish_reason': 'stop', 'logprobs': None}, id='run-4545fa92-d085-47f2-a1d4-1533a0141f4c-0', usage_metadata={'input_tokens': 30, 'output_tokens': 16, 'total_tokens': 46})

In [4]:
from approaches.Llama_direct_approach import LLamaDirectApproach

obj = LLamaDirectApproach(groq_model_name="mixtral-8x7b-32768", groq_api_key="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")

In [7]:
result = obj.run(history=None, overrides=None)

In [12]:
from groq import Groq

client = Groq(api_key="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")
completion = client.chat.completions.create(
    model="llama-3.2-1b-preview",
    messages=[{"role": "user", "content": "Write a passage of 500 words on AI Agents?"}],
    temperature=1,
    max_tokens=1024,
    top_p=1,
    stream=True,
    stop=None,
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")


**The Rise of AI Agents**

Artificial Intelligence (AI) agents have revolutionized the way we interact with machines. These agents are software entities that mimic human-like decision-making capabilities, often relying on machine learning algorithms to learn from experience and adapt to new situations. The concept of AI agents has been explored in various fields, including artificial intelligence, robotics, game development, and the broader realm of human-computer interaction.

The emergence of AI agents has several key characteristics that set them apart from traditional AI systems. Firstly, AI agents are autonomous, meaning they operate independently, making decisions without human intervention. However, this autonomy also comes with a degree of uncertainty, as the agent's decision-making process is often based on incomplete or ambiguous information.

Another defining feature of AI agents is their ability to learn from experience. Many AI agents are capable of unsupervised machine le

In [ ]:
# from core.modelhelper import num_tokens_from_messages_llama
from transformers import AutoTokenizer
def num_tokens_from_messages_llama(message: dict[str, str], model: str) -> int:
    """
    Calculate the number of tokens required to encode a message for the LLama model.
    
    Args:
        message (dict): The message to encode, represented as a dictionary.
        model (str): The name of the model to use for encoding (for LLama 3.2, you would use the appropriate LLama tokenizer).
    
    Returns:
        int: The total number of tokens required to encode the message.
    
    Example:
        message = {'role': 'user', 'content': 'Hello, how are you?'}
        model = 'llama-3.2'
        num_tokens_from_messages(message, model)
        output: 11
    """
    # Load the appropriate tokenizer for LLama
    tokenizer = AutoTokenizer.from_pretrained(model)

    # Initialize the token count with 2 tokens for the special tokens for "role" and "content"
    num_tokens = 2  # One for "role" and one for "content" in the dictionary
    
    for key, value in message.items():
        # Encode each content and sum the number of tokens
        num_tokens += len(tokenizer.encode(value))
    
    return num_tokens

ans = num_tokens_from_messages_llama(message={"role": "user", "content": "Write a passage of 500 words on AI Agents?"}, 
                                     model='meta-llama/Llama-3.2-1B-Instruct')

In [6]:
from core.modelhelper import num_tokens_from_messages
ans = num_tokens_from_messages(message={"role": "user", "content": "Write a passage of 500 words on AI Agents?"}, 
                                     model="gpt-4o")
ans

14

In [3]:
client = Groq(api_key="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")
completion = client.chat.completions.create(
    model="llama-3.2-1b-preview",
    messages=[{"role": "user", "content": "Write a passage of 500 words on AI Agents?"}],
    temperature=1,
    max_tokens=1024,
    top_p=1,
    stream=True,
    stop=None,
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")

14

In [39]:
import json
import re
import logging
from datetime import datetime, timedelta
from typing import Any, Sequence
from approaches.approach import Approach
from core.messagebuilder import MessageBuilder
from core.modelhelper import get_token_limit
from core.modelhelper import num_tokens_from_messages
from groq import Groq

class LlamaDirectApproach(Approach):

    SYSTEM = "system"
    USER = "user"
    ASSISTANT = "assistant"

    system_message_chat_conversation = """You are a Groq LLama system. Your persona is {systemPersona} who helps users interact with a Large Language Model. {response_length_prompt}
        User persona is {userPersona}. You are having a conversation with a user and you need to provide a response.    

        {follow_up_questions_prompt}
        {injected_prompt}
        """
    follow_up_questions_prompt_content = """
        Generate three very brief follow-up questions that the user would likely ask next about their previous chat context. Use triple angle brackets to reference the questions, e.g. <<<Are there exclusions for prescriptions?>>>. Try not to repeat questions that have already been asked.
        Only generate questions and do not generate any text before or after the questions, such as 'Next Questions'
        """
    query_prompt_template = """Below is a history of the conversation so far, and a new question asked by the user that needs to be answered.
        Generate a search query based on the conversation and the new question. Treat each search term as an individual keyword. Do not combine terms in quotes or brackets.
        Do not include cited sources e.g info or doc in the search query terms.
        Do not include any text inside [] or <<<>>> in the search query terms.
        Do not include any special characters like '+'.
        If the question is not in {query_term_language}, translate the question to {query_term_language} before generating the search query.
        If you cannot generate a search query, return just the number 0.
        """

    
    response_prompt_few_shots = []
    def __init__(
        self,
        llama_client,
        query_term_language: str,
        model_name: str,
    ):
        self.query_term_language = query_term_language
        self.chatgpt_token_limit = get_token_limit(model_name)
        self.model_name = model_name
        self.client = llama_client

    def run(self, history: Sequence[dict[str, str]], overrides: dict[str, Any], thought_chain: dict[str, Any]) -> Any:
        user_persona = overrides.get("user_persona", "")
        system_persona = overrides.get("system_persona", "")
        response_length = int(overrides.get("response_length") or 1024)

        user_q = 'Generate response for: ' + history[-1]["user"]
        thought_chain["user_query"] = history[-1]["user"]

        follow_up_questions_prompt = (
            self.follow_up_questions_prompt_content
            if overrides.get("suggest_followup_questions")
            else ""
        )

        system_message = self.system_message_chat_conversation.format(
            injected_prompt="",
            follow_up_questions_prompt=follow_up_questions_prompt,
            response_length_prompt=self.get_response_length_prompt_text(
                response_length
            ),
            userPersona=user_persona,
            systemPersona=system_persona,
        )

        messages = self.get_messages_from_history(
            system_message,
            "gpt-4o",
            history,
            history[-1]["user"] + "\n\n",
            self.response_prompt_few_shots,
            max_tokens=self.chatgpt_token_limit - 500
        )

        try:
            chat_completion = self.client.chat.completions.create(
                model=self.model_name,
                messages=messages,
                temperature=0.6,
                max_tokens=1024,
                top_p=1,
                stream=True,
                stop=None,
            ) 

            msg_to_display = '\n\n'.join([str(message) for message in messages])

            full_response = ""  
            for chunk in chat_completion:
                if hasattr(chunk, "choices") and len(chunk.choices) > 0 and hasattr(chunk.choices[0], "delta"):
                    delta_content = chunk.choices[0].delta.content
                    if delta_content:
                        full_response += delta_content  # Aggregate content

            # Yield the final structured response
            yield json.dumps({
                "data_points": {},
                "thoughts": f"Searched for:<br>{user_q}<br><br>Conversations:<br>" + msg_to_display.replace('\n', '<br>'),
                "thought_chain": thought_chain,
                "response": full_response,
            }) + "\n"
                
        except Exception as e:
            logging.error(f"Error in LlamaDirectApproach: {e}")
            yield json.dumps({"error": f"An error occurred while generating the completion. {e}"}) + "\n"
            return


In [13]:
client = Groq(api_key="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")
# completion = client.chat.completions.create(
#     model="llama-3.2-1b-preview",
#     messages=[{"role": "user", "content": "Write a passage of 500 words on AI Agents?"}],
#     temperature=1,
#     max_tokens=1024,
#     top_p=1,
#     stream=True,
#     stop=None,
# )

obj = LlamaDirectApproach(llama_client=client, query_term_language="english", model_name="llama-3.2-1b-preview")

In [47]:
import ast
def test_llama_direct_approach():
    # Instantiate the object
    obj = LlamaDirectApproach(
        llama_client=client,
        query_term_language="english",
        model_name="llama-3.2-1b-preview"
    )
    
    # Run the asynchronous method
    result_generator = obj.run(
        history=[{"system": "How can I help You today?", "user": "I want to know about hackathon!!"}],
        overrides={"temperature": 0.3},
        # citation_lookup={},
        thought_chain={"system": "i should search the web to gather the information about latest hackathons"}
    )

    # Consume the asynchronous generator
    for result in result_generator:
        print(result)
        result = ast.literal_eval(result)
        print(result['response'])

# Run the test function
result = test_llama_direct_approach()


{"data_points": {}, "thoughts": "Searched for:<br>Generate response for: I want to know about hackathon!!<br><br>Conversations:<br>{'role': 'system', 'content': 'You are a Groq LLama system. Your persona is  who helps users interact with a Large Language Model. Please provide a succinct answer. This means that your answer should be no more than 1024 tokens long.\\n        User persona is . You are having a conversation with a user and you need to provide a response.    \\n\\n        \\n        \\n        '}<br><br>{'role': 'user', 'content': 'I want to know about hackathon!!\\n\\n'}", "thought_chain": {"system": "i should search the web to gather the information about latest hackathons", "user_query": "I want to know about hackathon!!"}, "response": "Hackathons are exciting events where teams come together to solve real-world problems in a short period, typically 24-48 hours. They're a great way to learn, innovate, and build projects that can have a significant impact.\n\nWhat kind of 

In [43]:
result["data_points"]

TypeError: 'NoneType' object is not subscriptable